# 04.01 — Representation audit and row-level text features

## Goal
Validate the Phase 3 handoff, compare its two designated text representations, and create interpretable numeric features without using labels or market data.

## Reason
A stable schema and explicit representation decision are required before later model-specific extraction. This notebook never fabricates results when local data is absent.


## 1. Setup

### Goal
Load only the reusable Stage 1 utilities and establish repository-relative paths.

### Reason
Repository-relative paths make the workflow reproducible whether Jupyter starts at the root or inside a phase directory.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "3_text_preprocessing").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")


PROJECT_ROOT = find_project_root()
PHASE4_DIR = PROJECT_ROOT / "4_nlp_feature_extraction"
sys.path.insert(0, str(PHASE4_DIR))

from src.row_features import default_input_path, run_row_feature_pipeline

INPUT_PATH = default_input_path(PROJECT_ROOT)
FEATURE_PATH = PHASE4_DIR / "data" / "row_level_text_features.parquet"
AUDIT_PATH = PHASE4_DIR / "data" / "representation_audit.json"
print("Input:", INPUT_PATH)
print("Features:", FEATURE_PATH)
print("Audit:", AUDIT_PATH)


### Result
The printed paths identify the read-only Phase 3 input and the two Gitignored Phase 4 outputs.

### Interpretation
No data has been read or changed yet.

### Decision
Keep all generated artifacts under `4_nlp_feature_extraction/data`.


## 2. Validate the Phase 3 input

### Goal
Require `source_row_id`, `text_title_description`, and `Filtered_Text`; require at least one row; and require non-missing, unique IDs.

### Reason
Row count and duplicate-ID checks protect the one-row-per-article contract. A missing local file must stop execution rather than produce invented findings.


In [ ]:
if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Required Phase 3 input is unavailable: {INPUT_PATH}\n"
        "Run Phase 3 or place cryptovision_v1_preprocessed.parquet at this exact "
        "path, then restart and run all cells."
    )

# Loading, required-column validation, empty-input validation, the representation
# audit, feature construction, and saving are performed by one tested pipeline.
features, audit = run_row_feature_pipeline(INPUT_PATH, FEATURE_PATH, AUDIT_PATH)


### Result
On real local data, the cell completes only after schema, row count, missing ID, and duplicated ID validation pass. If the Parquet is absent, the message gives its exact expected path.

### Interpretation
The Phase 3 file remains read-only; only derived Phase 4 outputs are written.

### Decision
Treat any validation error as a contract problem and resolve it upstream rather than weakening checks.


## 3. Compare representations

### Goal
Review empty counts, total character/word counts, medians, 95th percentiles, maxima, and very-short counts for both candidate representations.

### Reason
Length and coverage differ across representations. The compact report makes that difference visible without selecting a representation using outcomes.


In [ ]:
import pandas as pd

rows = []
for representation, metrics in audit["representations"].items():
    rows.append({"representation": representation, **metrics})
audit_table = pd.DataFrame(rows).set_index("representation")
audit_table


### Result
The table above is computed from the real handoff and the same compact content is saved to `data/representation_audit.json`.

### Interpretation
“Very short” means non-empty text containing fewer than five words. Missing values and whitespace-only strings count as empty. Percentiles include empty rows so coverage remains visible.

### Decision
Use `text_title_description` for Stage 1 row features. This is a content/provenance decision—not an outcome-driven choice—and both representations remain available for later controlled experiments.


## 4. Inspect row-level features

### Goal
Confirm that the saved table contains one identifier and numeric, explainable text measurements.

### Reason
Simple features provide transparent baselines and diagnostics before model-specific representations.


In [ ]:
print("Feature rows:", len(features))
print("Unique IDs:", features["source_row_id"].nunique())
print("Output written:", FEATURE_PATH.is_file())
features.head()


### Result
`row_level_text_features.parquet` contains length, word/digit counts, digit and uppercase ratios, percentage/currency mentions, punctuation measurements, and documented finance keyword presence flags.

### Interpretation
Ratios use zero for empty denominators. Keyword columns are case-insensitive presence indicators, not sentiment or outcome labels. Sentiment, OHLCV, market-move, and outcome fields are never read by feature construction.

### Decision
Join downstream data only through `source_row_id`. Do not treat the identifier as a predictive numeric feature.


## 5. Stage boundary

### Goal
Record what this notebook intentionally does not do.

### Reason
Feature families should be added in separately testable, leakage-safe experiments.

### Code
There is no TF-IDF, BERT, FinBERT, PCA, or SVD code in this notebook.

### Result
Stage 1 ends with a validated audit and explainable row features.

### Interpretation
Model-specific text matrices and dimensionality reduction require separate train/test fitting decisions.

### Decision
Implement those methods only in later Phase 4 stages as documented in the Phase 4 README.
